# Kolumbus GTFS + real-time (SIRI ET): exploration and first cleaning pass

**Goal:** get a clean table of *bus stop visits that really happened* (stop, time, delay), and compare it with what the static timetable says should have happened.

| Part | What it does |
|---|---|
| 0 | Setup and one-place config |
| 1 | Peek at every parquet file (replaces the many repeated schema/metadata cells) |
| 2 | Static GTFS: routes, route types, shape → trip → stops chain |
| 3 | Service alerts |
| 4 | Real-time `stop_visits` (ET): what the rows really are + data-quality profile |
| 5 | **Expected stops per day** from the static timetable |
| 6 | **Cleaning** real-time visits: dedupe snapshots, parse times, bus-only, real stops only |
| 7 | **Expected vs observed**: how many times buses actually came to a stop |
| 8 | Save clean tables for the next step |

**Key idea:** `stop_visits` is a stream of snapshots. The same visit (one bus at one stop on one day) shows up many times with updated predictions, so raw row counts are *not* visit counts. Part 6 fixes that.

## 0. Setup

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

DATA_DIR = Path(".")          # folder that holds the .parquet files
OUT_DIR = Path("clean")       # cleaned outputs are written here
OUT_DIR.mkdir(exist_ok=True)
LOCAL_TZ = "Europe/Oslo"      # matches agency_timezone in gtfs_agency

# Which service dates of the real-time feed to analyse. None = every date found in the feed.
ET_DATES = None               # e.g. ["2026-10-24"]

def pq_path(name):
    return DATA_DIR / f"{name}.parquet"

## 1. Peek at every file

One loop instead of one cell per file. For each file: rows, columns, and the snapshot range for the static GTFS files.

In [3]:
FILES = [
    "gtfs_agency", "gtfs_routes", "gtfs_trips", "gtfs_stop_times", "gtfs_stops",
    "gtfs_calendar", "gtfs_calendar_dates", "gtfs_shapes",
    "service_alerts", "stop_visits",
]

rows = []
for name in FILES:
    pf = pq.ParquetFile(pq_path(name))
    rows.append({
        "file": name,
        "rows": pf.metadata.num_rows,
        "cols": pf.metadata.num_columns,
        "row_groups": pf.metadata.num_row_groups,
        "columns": ", ".join(pf.schema_arrow.names),
    })
overview = pd.DataFrame(rows)
display(overview[["file", "rows", "cols", "row_groups"]])

# Full column lists, if you want to read them
for r in rows:
    print(f"{r['file']:<20} {r['columns']}")

,file,rows,cols,row_groups
0,gtfs_agency,3,6,1
1,gtfs_routes,318,7,1
2,gtfs_trips,13833,9,1
3,gtfs_stop_times,418837,11,1
4,gtfs_stops,7676,12,1
5,gtfs_calendar,223,12,1
6,gtfs_calendar_dates,2722,5,1
7,gtfs_shapes,614593,7,1
8,service_alerts,53,17,1
9,stop_visits,9989935,26,100


gtfs_agency          agency_id, agency_name, agency_url, agency_timezone, snapshot_time, source_file
gtfs_routes          agency_id, route_id, route_short_name, route_long_name, route_type, snapshot_time, source_file
gtfs_trips           route_id, trip_id, service_id, trip_headsign, direction_id, shape_id, boarding_type, snapshot_time, source_file
gtfs_stop_times      trip_id, stop_id, arrival_time, departure_time, stop_sequence, stop_headsign, pickup_type, drop_off_type, shape_dist_traveled, snapshot_time, source_file
gtfs_stops           stop_id, stop_name, stop_lat, stop_lon, stop_desc, location_type, parent_station, wheelchair_boarding, vehicle_type, platform_code, snapshot_time, source_file
gtfs_calendar        service_id, monday, tuesday, wednesday, thursday, friday, saturday, sunday, start_date, end_date, snapshot_time, source_file
gtfs_calendar_dates  service_id, date, exception_type, snapshot_time, source_file
gtfs_shapes          shape_id, shape_pt_sequence, shape_pt_lat, sha

**Things worth knowing**

- All GTFS columns are stored as **strings** (dates like `20261024`, `stop_sequence`, flags like `"1"`). Convert before doing maths or comparisons.
- `snapshot_time` / `source_file` are added by your pipeline, not part of GTFS. Check that the static tables hold a single snapshot, otherwise rows would be duplicated:

In [4]:
for name in ["gtfs_agency", "gtfs_routes", "gtfs_trips", "gtfs_stop_times", "gtfs_stops", "gtfs_calendar", "gtfs_calendar_dates"]:
    snaps = pd.read_parquet(pq_path(name), columns=["snapshot_time"])["snapshot_time"]
    print(f"{name:<20} distinct snapshots: {snaps.nunique()}  ({snaps.min()} → {snaps.max()})")

gtfs_agency          distinct snapshots: 1  (2026-10-24 20:00:12+00:00 → 2026-10-24 20:00:12+00:00)
gtfs_routes          distinct snapshots: 1  (2026-10-24 20:00:12+00:00 → 2026-10-24 20:00:12+00:00)
gtfs_trips           distinct snapshots: 1  (2026-10-24 20:00:12+00:00 → 2026-10-24 20:00:12+00:00)
gtfs_stop_times      distinct snapshots: 1  (2026-10-24 20:00:12+00:00 → 2026-10-24 20:00:12+00:00)
gtfs_stops           distinct snapshots: 1  (2026-10-24 20:00:12+00:00 → 2026-10-24 20:00:12+00:00)
gtfs_calendar        distinct snapshots: 1  (2026-10-24 20:00:12+00:00 → 2026-10-24 20:00:12+00:00)
gtfs_calendar_dates  distinct snapshots: 1  (2026-10-24 20:00:12+00:00 → 2026-10-24 20:00:12+00:00)


## 2. Static GTFS (the timetable)

In [5]:
agency = pd.read_parquet(pq_path("gtfs_agency"))
routes = pd.read_parquet(pq_path("gtfs_routes"))
trips = pd.read_parquet(pq_path("gtfs_trips"))
stops = pd.read_parquet(pq_path("gtfs_stops"))
calendar = pd.read_parquet(pq_path("gtfs_calendar"))
calendar_dates = pd.read_parquet(pq_path("gtfs_calendar_dates"))
stop_times = pd.read_parquet(
    pq_path("gtfs_stop_times"),
    columns=["trip_id", "arrival_time", "departure_time", "stop_id", "stop_sequence"],
)

display(agency[["agency_id", "agency_name", "agency_timezone"]])
print(f"{len(routes):,} routes | {len(trips):,} trips | {len(stop_times):,} stop_times | {len(stops):,} stops")

,agency_id,agency_name,agency_timezone
0,KOL:Authority:26,Norled AS,Europe/Oslo
1,KOL:Authority:30,Fjord1 ASA,Europe/Oslo
2,KOL:Authority:8,Kolumbus,Europe/Oslo


318 routes | 13,833 trips | 418,837 stop_times | 7,676 stops


### 2.1 What is `route_type`?

These are **extended GTFS route types** (the 3-digit / 4-digit codes), not the basic 0–7 ones. The usual meanings (double-check against the official list if it matters for a result):

| Code | Meaning |
|---|---|
| 704 | Local bus service |
| 705 | Night bus service |
| 712 | School bus service |
| 1004 | Local car ferry |
| 1008 | Local passenger ferry |
| 1014 | High-speed passenger ferry |

In [6]:
ROUTE_TYPE_NAMES = {
    "704": "Local bus", "705": "Night bus", "712": "School bus",
    "1004": "Local car ferry", "1008": "Local passenger ferry", "1014": "High-speed passenger ferry",
}
routes["route_type_name"] = routes["route_type"].map(ROUTE_TYPE_NAMES).fillna("unknown")

(routes.groupby(["route_type", "route_type_name"]).size()
       .rename("n_routes").reset_index().sort_values("n_routes", ascending=False))

,route_type,route_type_name,n_routes
5,712,School bus,179
3,704,Local bus,105
4,705,Night bus,13
2,1014,High-speed passenger ferry,12
0,1004,Local car ferry,8
1,1008,Local passenger ferry,1


### 2.2 Is 712 really "school bus"? (your Uis Ost check, made repeatable)

Your earlier check looked at routes 6, 7 and 8 and found they are type 704. Instead of guessing route numbers, compare the **stops** each route type serves. School routes (`SK….` short names, no long name) should serve a different set of stops than ordinary local routes, with some overlap at big terminals.

In [7]:
# route_type -> set of stop_ids it serves
rt = routes[["route_id", "route_type"]]
tr = trips[["trip_id", "route_id"]].merge(rt, on="route_id", how="left")
st = stop_times[["trip_id", "stop_id"]].merge(tr[["trip_id", "route_type"]], on="trip_id", how="left")
stops_by_type = st.groupby("route_type")["stop_id"].agg(lambda s: set(s))

local, school = stops_by_type.get("704", set()), stops_by_type.get("712", set())
print(f"stops served by 704 (local bus):  {len(local):,}")
print(f"stops served by 712 (school bus): {len(school):,}")
print(f"stops served by BOTH:             {len(local & school):,}")
print(f"stops served ONLY by 712:         {len(school - local):,}")

# Look up a stop by name, e.g. Uis
name_to_ids = stops[stops["stop_name"].str.contains("Uis", case=False, na=False)][["stop_id", "stop_name"]]
display(name_to_ids.assign(
    served_by=lambda d: d["stop_id"].map(lambda sid: sorted(t for t, s in stops_by_type.items() if sid in s))
))

stops served by 704 (local bus):  3,821
stops served by 712 (school bus): 3,074
stops served by BOTH:             2,202
stops served ONLY by 712:         872


,stop_id,stop_name,served_by
1097,NSR:Quay:44969,UiS nord,"[704, 705, 712]"
1098,NSR:Quay:44970,UiS nord,"[704, 705]"
1108,NSR:Quay:44984,UiS øst,"[704, 705, 712]"
1110,NSR:Quay:44986,UiS øst,"[704, 705]"
4930,NSR:StopPlace:25943,UiS nord,[]
4937,NSR:StopPlace:25953,UiS øst,[]


### 2.3 Shape → trip → route → stops

A `shape_id` is only a line on the map. The chain is:

`shape_id` → **trips** (gives `route_id` and `trip_id`) → **stop_times** (gives the ordered stops) → **stops** (gives names and coordinates).

Your example turned into a reusable function:

In [8]:
def describe_shape(shape_id):
    t = trips[trips["shape_id"] == shape_id]
    if t.empty:
        print("No trips use this shape_id.")
        return None
    r = routes[routes["route_id"].isin(t["route_id"])][["route_id", "route_short_name", "route_long_name", "route_type", "route_type_name"]]
    print(f"{len(t):,} trips use this shape, on route(s):")
    display(r)

    one_trip = t["trip_id"].iloc[0]
    s = stop_times[stop_times["trip_id"] == one_trip].copy()
    s["stop_sequence"] = s["stop_sequence"].astype(int)
    out = (s.sort_values("stop_sequence")
             .merge(stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]], on="stop_id", how="left")
             [["stop_sequence", "stop_id", "stop_name", "arrival_time", "departure_time"]])
    print(f"Stops for example trip {one_trip}:")
    return out

describe_shape("KOL:JourneyPattern:1001_Inbound_251008118370718_250903083108828")

248 trips use this shape, on route(s):


,route_id,route_short_name,route_long_name,route_type,route_type_name
2,KOL:Line:8_1001,1,,704,Local bus


Stops for example trip KOL:ServiceJourney:1001_251008118370980_2001:


,stop_sequence,stop_id,stop_name,arrival_time,departure_time
0,1,NSR:Quay:103376,Sandnes bussterminal Ruten,05:30:00,05:30:00
1,2,NSR:Quay:109015,Olav Vs plass,05:32:00,05:32:00
2,3,NSR:Quay:111020,Jernbaneveien,05:33:00,05:33:00
3,4,NSR:Quay:47663,Strandgata,05:35:00,05:35:00
4,5,NSR:Quay:109013,Strandgata nord,05:37:00,05:37:00
5,6,NSR:Quay:49004,Gamle Postvei,05:39:00,05:39:00
6,7,NSR:Quay:45344,Seljeveien,05:40:00,05:40:00
7,8,NSR:Quay:47836,Kvadrat,05:41:00,05:41:00
8,9,NSR:Quay:48301,Porsmyrveien,05:42:00,05:42:00
9,10,NSR:Quay:46365,Jakob Askelands vei,05:43:00,05:43:00


### 2.4 Static-table sanity checks

Cheap checks that catch problems early. Anything that prints a non-zero "problem" count deserves a look.

In [9]:
def check(label, bad, total=None):
    tail = f" of {total:,}" if total is not None else ""
    print(f"{'OK ' if bad == 0 else 'LOOK'}  {label}: {bad:,}{tail}")

check("duplicate trip_id in trips", trips["trip_id"].duplicated().sum(), len(trips))
check("duplicate stop_id in stops", stops["stop_id"].duplicated().sum(), len(stops))
check("stop_times with trip_id not in trips", (~stop_times["trip_id"].isin(trips["trip_id"])).sum(), len(stop_times))
check("stop_times with stop_id not in stops", (~stop_times["stop_id"].isin(stops["stop_id"])).sum(), len(stop_times))
check("trips with route_id not in routes", (~trips["route_id"].isin(routes["route_id"])).sum(), len(trips))
check("trips with service_id in neither calendar nor calendar_dates",
      (~trips["service_id"].isin(set(calendar["service_id"]) | set(calendar_dates["service_id"]))).sum(), len(trips))
check("stops with missing coordinates", (stops["stop_lat"].isna() | (stops["stop_lat"].astype(str).str.strip() == "")).sum(), len(stops))
print("calendar service dates:", calendar["start_date"].min(), "→", calendar["end_date"].max())
print("exception types in calendar_dates:", calendar_dates["exception_type"].value_counts().to_dict(), "(1 = service added, 2 = service removed)")

OK   duplicate trip_id in trips: 0 of 13,833
OK   duplicate stop_id in stops: 0 of 7,676
OK   stop_times with trip_id not in trips: 0 of 418,837
OK   stop_times with stop_id not in stops: 0 of 418,837
OK   trips with route_id not in routes: 0 of 13,833
OK   trips with service_id in neither calendar nor calendar_dates: 0 of 13,833
OK   stops with missing coordinates: 0 of 7,676
calendar service dates: 20261016 → 20270403
exception types in calendar_dates: {'2': 1942, '1': 780} (1 = service added, 2 = service removed)


## 3. Service alerts

Fixes to the original cells: comments said `alert_type`/`'LINE'` but the column is `affect_type` with values `line` / `journey`. Alerts are also **snapshots**, so the same situation can repeat across snapshots.

In [10]:
alerts = pd.read_parquet(pq_path("service_alerts"))
print(f"{len(alerts):,} alert rows, {alerts['situation_number'].nunique():,} distinct situations")
print(alerts["affect_type"].value_counts().to_string(), "\n")
print(alerts["severity"].value_counts().to_string(), "\n")
print(alerts["summary"].value_counts().head(10).to_string())

# one row per situation + target, keeping the newest snapshot
alerts_latest = (alerts.sort_values("snapshot_time")
                       .drop_duplicates(["situation_number", "line_ref", "journey_ref"], keep="last"))
print(f"\nafter de-duplicating snapshots: {len(alerts_latest):,} rows")
display(alerts_latest[alerts_latest["affect_type"] == "journey"][
    ["snapshot_time", "summary", "severity", "line_ref", "journey_ref", "valid_start", "valid_end"]
])

53 alert rows, 10 distinct situations
affect_type
line       44
journey     9 

severity
normal      41
noImpact     6
severe       6 

summary
Mulige forsinkelser pga. vanskelige værforhold    41
Forsinket på grunn av mye trafikk                  4
Trafikkmelding                                     3
Redusert kapasitet                                 3
Forsinket på grunn av tekniske problemer           1
Tur avsluttes før endeholdeplass                   1

after de-duplicating snapshots: 11 rows


,snapshot_time,summary,severity,line_ref,journey_ref,valid_start,valid_end
9,2026-10-24 10:59:35+00:00,Redusert kapasitet,noImpact,KOL:Line:26_5958,NaN,2026-10-24T13:00:00+02:00,2026-10-24T13:12:00+02:00
16,2026-10-24 11:51:39+00:00,Forsinket på grunn av tekniske problemer,severe,KOL:Line:8_1006,KOL:ServiceJourney:1006_261005064436180_3024,2026-10-24T00:00:00+02:00,2026-10-24T15:57:00+02:00
28,2026-10-24 15:57:00+00:00,Forsinket på grunn av mye trafikk,severe,KOL:Line:8_2010,KOL:ServiceJourney:2010_260923062817478_4014,2026-10-24T00:00:00+02:00,2026-10-24T18:06:00+02:00
32,2026-10-24 16:17:01+00:00,Forsinket på grunn av mye trafikk,severe,KOL:Line:8_6100,KOL:ServiceJourney:6100_250903083239593_4017,2026-10-24T00:00:00+02:00,2026-10-24T19:28:00+02:00
43,2026-10-24 19:39:18+00:00,Redusert kapasitet,noImpact,KOL:Line:26_5958,NaN,2026-10-24T21:40:00+02:00,2026-10-24T21:53:00+02:00
50,2026-10-24 23:24:37+00:00,Tur avsluttes før endeholdeplass,severe,KOL:Line:8_1104,KOL:ServiceJourney:1104_251007084239511_3070,2026-10-24T00:00:00+02:00,2026-10-25T02:31:00+02:00


Alert `journey_ref` and `line_ref` use the same ID style as GTFS `trip_id` and `route_id`. Let's verify that, since the whole comparison in Part 7 depends on it:

In [11]:
j = alerts_latest["journey_ref"].dropna().unique()
l = alerts_latest["line_ref"].dropna().unique()
print(f"alert journey_ref found in gtfs trip_id : {np.isin(j, trips['trip_id']).sum()}/{len(j)}")
print(f"alert line_ref found in gtfs route_id   : {np.isin(l, routes['route_id']).sum()}/{len(l)}")

alert journey_ref found in gtfs trip_id : 4/4
alert line_ref found in gtfs route_id   : 9/9


## 4. Real-time `stop_visits` (SIRI Estimated Timetable)

### 4.1 Load only what we need

The file has ~10M rows × 26 columns. Loading everything with `to_pandas()` is wasteful, so we read just the columns used later. We skip `stop_name`/`destination` (mostly empty here; the GTFS stop name is more reliable), `source_file` and `recorded_at`.

Column meaning (SIRI):

- `data_frame_ref` = the **service date** of the journey
- `journey_ref` = matches GTFS `trip_id`; `line_ref` = matches GTFS `route_id`; `stop_ref` = matches GTFS `stop_id`
- `aimed_*` = timetable time, `expected_*` = current prediction, `actual_*` = what really happened (filled in only after it happened)
- `snapshot_time` = when *we* downloaded this version of the row

In [12]:
ET_COLS = [
    "snapshot_time", "data_frame_ref", "journey_ref", "line_ref", "direction_ref", "vehicle_mode",
    "monitored", "journey_cancelled", "call_cancelled", "call_type",
    "stop_order", "stop_ref",
    "aimed_arrival", "expected_arrival", "actual_arrival",
    "aimed_departure", "expected_departure", "actual_departure",
    "arrival_status", "departure_status", "prediction_inaccurate",
]
et_raw = pd.read_parquet(pq_path("stop_visits"), columns=ET_COLS)
print(f"{len(et_raw):,} raw rows | {et_raw.memory_usage(deep=True).sum() / 1e6:,.0f} MB in memory")
et_raw.head(3)

9,989,935 raw rows | 4,513 MB in memory


,snapshot_time,data_frame_ref,journey_ref,line_ref,direction_ref,vehicle_mode,monitored,journey_cancelled,call_cancelled,call_type,stop_order,stop_ref,aimed_arrival,expected_arrival,actual_arrival,aimed_departure,expected_departure,actual_departure,arrival_status,departure_status,prediction_inaccurate
0,2026-10-24 00:00:15+00:00,2026-10-24,KOL:ServiceJourney:1130_250903083242106_4040,KOL:Line:30_1130,inbound,ferry,True,False,False,estimated,1,NSR:Quay:99269,NaN,NaN,NaN,2026-10-24T18:30:00+02:00,2026-10-24T18:30:00+02:00,NaN,NaN,NaN,False
1,2026-10-24 00:00:15+00:00,2026-10-24,KOL:ServiceJourney:1130_250903083242106_4040,KOL:Line:30_1130,inbound,ferry,True,False,False,estimated,2,NSR:Quay:99704,2026-10-24T18:54:00+02:00,2026-10-24T18:54:00+02:00,NaN,NaN,NaN,NaN,NaN,NaN,False
2,2026-10-24 00:00:15+00:00,2026-10-24,KOL:ServiceJourney:5958_250903083240300_8028,KOL:Line:26_5958,inbound,ferry,True,False,False,estimated,1,NSR:Quay:47191,NaN,NaN,NaN,2026-10-24T16:40:00+02:00,2026-10-24T16:40:00+02:00,NaN,NaN,NaN,False


### 4.2 Why raw row counts are misleading

Your earlier cell reported 1,290 distinct stop names. That is a *name* count, and `stop_name` is mostly empty in the feed (the first row has `NaN`). Counting by `stop_ref` is the right way. Also, one visit appears in many snapshots:

In [13]:
KEY = ["data_frame_ref", "journey_ref", "stop_order"]     # one real visit = this key

print("service dates in feed:", sorted(et_raw["data_frame_ref"].dropna().unique()))
print(f"distinct stop_ref  : {et_raw['stop_ref'].nunique():,}")

per_visit = et_raw.groupby(KEY).size()
print(f"\nunique visits (journey × stop × date): {len(per_visit):,}")
print(f"raw rows                              : {len(et_raw):,}")
print("snapshots per visit:")
print(per_visit.describe().round(1).to_string())

service dates in feed: ['2026-10-23', '2026-10-24', '2026-10-25', '2026-10-26', '2026-10-27']
distinct stop_ref  : 2,247

unique visits (journey × stop × date): 91,273
raw rows                              : 9,989,935
snapshots per visit:
count    91273.0
mean       109.5
std         67.2
min          2.0
25%         93.0
50%        105.0
75%        125.0
max       1428.0


### 4.3 Data-quality profile: what values exist, and how empty are the columns?

In [14]:
print("Share of nulls per column (%):")
print((et_raw.isna().mean() * 100).round(1).sort_values(ascending=False).to_string())

for c in ["vehicle_mode", "call_type", "arrival_status", "departure_status",
          "monitored", "journey_cancelled", "call_cancelled", "prediction_inaccurate"]:
    print(f"\n--- {c} ---")
    print(et_raw[c].value_counts(dropna=False).to_string())

Share of nulls per column (%):
journey_cancelled        95.0
prediction_inaccurate    95.0
call_cancelled           94.4
actual_arrival           73.4
actual_departure         72.5
departure_status         35.0
arrival_status           32.6
expected_departure       31.6
expected_arrival         30.8
aimed_departure           4.3
aimed_arrival             4.3
snapshot_time             0.0
data_frame_ref            0.0
direction_ref             0.0
line_ref                  0.0
journey_ref               0.0
stop_ref                  0.0
monitored                 0.0
stop_order                0.0
call_type                 0.0
vehicle_mode              0.0

--- vehicle_mode ---
vehicle_mode
bus      9491945
ferry     495561
NaN         2429

--- call_type ---
call_type
estimated    7207575
recorded     2782360

--- arrival_status ---
arrival_status
delayed      6246114
NaN          3257238
early         420931
cancelled      64420
onTime          1232

--- departure_status ---
departure_st

**Read this before Part 6.** The value counts above decide the definition of "a bus came to the stop". The cleaning below uses `actual_arrival` / `actual_departure` being filled in, as the proof. If those columns turn out to be almost empty in your data, we must switch to a looser rule (see 6.4).

## 5. Expected stops per day from the static timetable

A trip runs on a date if its `service_id` is **active** on that date:

1. in `calendar`: date between `start_date` and `end_date`, and that weekday's flag is `1`
2. plus services **added** by `calendar_dates` (`exception_type = 1`)
3. minus services **removed** by `calendar_dates` (`exception_type = 2`), e.g. holidays

Then every row in `stop_times` of those trips is one **scheduled stop event**.

In [ ]:
BUS_ROUTE_TYPES = {"3"} | {str(x) for x in range(700, 800)}     # 7xx = bus family in extended GTFS
routes["is_bus"] = routes["route_type"].isin(BUS_ROUTE_TYPES)

for c in ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]:
    calendar[c] = calendar[c].astype(str).str.strip().eq("1")


def active_service_ids(service_date):
    """service_ids running on `service_date` (pd.Timestamp or 'YYYY-MM-DD')."""
    d = pd.Timestamp(service_date)
    ds = d.strftime("%Y%m%d")
    weekday = d.day_name().lower()
    base = calendar[(calendar["start_date"] <= ds) & (calendar["end_date"] >= ds) & calendar[weekday]]["service_id"]
    exc = calendar_dates[calendar_dates["date"] == ds]
    added = exc[exc["exception_type"].astype(str).str.strip() == "1"]["service_id"]
    removed = exc[exc["exception_type"].astype(str).str.strip() == "2"]["service_id"]
    return (set(base) | set(added)) - set(removed)


def gtfs_time_to_ts(service_date, hhmmss):
    """GTFS times are local, can pass 24:00:00, and count from 'noon minus 12h' of the service day."""
    anchor = pd.Timestamp(f"{pd.Timestamp(service_date).date()} 12:00", tz=LOCAL_TZ) - pd.Timedelta(hours=12)
    return anchor + pd.to_timedelta(hhmmss)


trip_info = trips[["trip_id", "route_id", "service_id"]].merge(
    routes[["route_id", "route_short_name", "route_type", "route_type_name", "is_bus"]], on="route_id", how="left"
)
st_all = stop_times.copy()
st_all["stop_sequence"] = st_all["stop_sequence"].astype(int)
st_all = st_all.sort_values(["trip_id", "stop_sequence"])
# occurrence number: handles loops where a trip calls at the same stop twice
st_all["occ"] = st_all.groupby(["trip_id", "stop_id"]).cumcount()
# position inside the trip (1, 2, 3 ...), to compare with the real-time stop_order
st_all["stop_rank"] = st_all.groupby("trip_id").cumcount() + 1


def expected_events(service_date):
    sids = active_service_ids(service_date)
    t = trip_info[trip_info["service_id"].isin(sids)]
    ev = st_all.merge(t, on="trip_id", how="inner")
    ev["service_date"] = pd.Timestamp(service_date).strftime("%Y-%m-%d")
    ev["sched_arrival"] = gtfs_time_to_ts(service_date, ev["arrival_time"])
    ev["sched_departure"] = gtfs_time_to_ts(service_date, ev["departure_time"])
    return ev

### 5.1 Expected number of stops per day

In [ ]:
et_dates = sorted(et_raw["data_frame_ref"].dropna().unique()) if ET_DATES is None else ET_DATES
expected = pd.concat([expected_events(d) for d in et_dates], ignore_index=True)

summary = (expected.groupby(["service_date", "route_type_name"])
                   .agg(trips=("trip_id", "nunique"), scheduled_stop_events=("stop_id", "size"), distinct_stops=("stop_id", "nunique"))
                   .reset_index())
display(summary)

print("BUS ONLY:")
display(expected[expected["is_bus"]].groupby("service_date")
        .agg(trips=("trip_id", "nunique"), scheduled_stop_events=("stop_id", "size"), distinct_stops=("stop_id", "nunique")))

### 5.2 Expected visits per stop per day

`expected_per_stop` answers "how many times *should* a bus call at stop X on date D".

In [ ]:
expected_per_stop = (expected[expected["is_bus"]]
    .groupby(["service_date", "stop_id"]).size().rename("expected_visits").reset_index()
    .merge(stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]], on="stop_id", how="left"))
display(expected_per_stop.sort_values("expected_visits", ascending=False).head(15))

## 6. Cleaning the real-time visits

Steps:

1. Keep the chosen service dates; drop rows without a key.
2. **De-duplicate snapshots**: one row per visit. Rows are sorted by `snapshot_time`, and for each visit we take the last non-null value of every column. This matters because `actual_*` is filled in late, so the newest snapshot sometimes lacks it while an older/newer one has it.
3. Parse timestamps to proper local datetimes.
4. Check the IDs link to GTFS (trip, stop, route).
5. Keep **buses** and **real stops** only.
6. Mark each visit's status (`cancelled` / `observed` / `unconfirmed`) and compute delay.

In [ ]:
# 1. filter
et = et_raw[et_raw["data_frame_ref"].isin(et_dates)].dropna(subset=KEY + ["stop_ref"]).copy()

# 2. one row per visit: newest snapshot, filling blanks from other snapshots of the same visit
et = et.sort_values("snapshot_time")
et_visits = et.groupby(KEY, sort=False).last().reset_index()
et_visits["n_snapshots"] = et.groupby(KEY, sort=False).size().to_numpy()
print(f"raw rows {len(et_raw):,} → de-duplicated visits {len(et_visits):,}")

In [ ]:
# 3. parse timestamps (ISO strings with offsets, some with nanoseconds) → tz-aware local time
TIME_COLS = ["aimed_arrival", "expected_arrival", "actual_arrival",
             "aimed_departure", "expected_departure", "actual_departure"]
for c in TIME_COLS:
    et_visits[c] = pd.to_datetime(et_visits[c], format="ISO8601", utc=True, errors="coerce").dt.tz_convert(LOCAL_TZ)

et_visits["snapshot_time"] = et_visits["snapshot_time"].dt.tz_convert(LOCAL_TZ)
print((et_visits[TIME_COLS].notna().mean() * 100).round(1).rename("% filled").to_string())

In [ ]:
# 4. do the IDs link to GTFS?
trip_ids, stop_ids, route_ids = set(trips["trip_id"]), set(stops["stop_id"]), set(routes["route_id"])
print(f"journey_ref in GTFS trips : {et_visits['journey_ref'].isin(trip_ids).mean():.1%}")
print(f"stop_ref in GTFS stops    : {et_visits['stop_ref'].isin(stop_ids).mean():.1%}")
print(f"line_ref in GTFS routes   : {et_visits['line_ref'].isin(route_ids).mean():.1%}")

# 5. buses and real stops only
et_visits = et_visits.merge(
    routes[["route_id", "route_short_name", "route_type", "route_type_name", "is_bus"]],
    left_on="line_ref", right_on="route_id", how="left",
).drop(columns="route_id")
et_visits = et_visits.merge(
    stops[["stop_id", "stop_name", "stop_lat", "stop_lon", "parent_station"]],
    left_on="stop_ref", right_on="stop_id", how="left",
).drop(columns="stop_id")

print()
print(pd.crosstab(et_visits["vehicle_mode"], et_visits["route_type_name"]).to_string())   # sanity check

before = len(et_visits)
et_visits = et_visits[et_visits["is_bus"].fillna(False) & et_visits["stop_name"].notna()].copy()
print(f"\nkept {len(et_visits):,} of {before:,} visits (bus route + stop exists in GTFS)")

### 6.4 What counts as "the bus came to the stop"?

| `visit_status` | Rule |
|---|---|
| `cancelled` | journey or call cancelled |
| `observed` | not cancelled **and** `actual_arrival` or `actual_departure` is filled in |
| `unconfirmed` | not cancelled, no actual time (could be future, or the feed never reported it) |

Only `observed` rows go into the "pure" table. `unconfirmed` is kept so you can see how big the gap is.

In [ ]:
has_actual = et_visits["actual_arrival"].notna() | et_visits["actual_departure"].notna()
cancelled = et_visits["journey_cancelled"].fillna(False) | et_visits["call_cancelled"].fillna(False)

et_visits["visit_status"] = np.select([cancelled, has_actual], ["cancelled", "observed"], default="unconfirmed")

# Time the bus was at the stop: arrival if we have it, otherwise departure (first stop has no arrival)
et_visits["visit_time"]  = et_visits["actual_arrival"].fillna(et_visits["actual_departure"])
et_visits["aimed_time"]  = np.where(et_visits["actual_arrival"].notna(), et_visits["aimed_arrival"], et_visits["aimed_departure"])
et_visits["aimed_time"]  = pd.to_datetime(et_visits["aimed_time"], utc=True).dt.tz_convert(LOCAL_TZ)
et_visits["delay_s"]     = (et_visits["visit_time"] - et_visits["aimed_time"]).dt.total_seconds()
et_visits["delay_arrival_s"]   = (et_visits["actual_arrival"] - et_visits["aimed_arrival"]).dt.total_seconds()
et_visits["delay_departure_s"] = (et_visits["actual_departure"] - et_visits["aimed_departure"]).dt.total_seconds()

# Flag implausible delays instead of silently dropping them (> 3 h late or > 30 min early)
et_visits["delay_outlier"] = (et_visits["delay_s"] > 3 * 3600) | (et_visits["delay_s"] < -30 * 60)

display(et_visits["visit_status"].value_counts().rename("visits").to_frame()
        .assign(share=lambda d: (d["visits"] / d["visits"].sum()).round(3)))
print("delay outliers among observed:", int((et_visits["delay_outlier"] & (et_visits["visit_status"] == "observed")).sum()))

## 7. Expected vs observed: how many times did buses come to a stop?

Each scheduled stop event is matched to the real-time visit by `(service date, trip, stop, occurrence)`. We use the *occurrence number* rather than `stop_sequence`/`stop_order` directly, because the two numberings are not guaranteed to line up. First, check that IDs match at all:

In [ ]:
et_visits["occ"] = et_visits.sort_values("stop_order").groupby(["data_frame_ref", "journey_ref", "stop_ref"]).cumcount()

exp_bus = expected[expected["is_bus"]]
sched_trips = set(zip(exp_bus["service_date"], exp_bus["trip_id"]))
et_trips = set(zip(et_visits["data_frame_ref"], et_visits["journey_ref"]))
print(f"scheduled bus trips : {len(sched_trips):,}")
print(f"trips seen in ET    : {len(et_trips):,}")
print(f"in both             : {len(sched_trips & et_trips):,}")
print(f"ET only (not in static timetable, e.g. added trips): {len(et_trips - sched_trips):,}")
print(f"timetable only (no real-time data at all)         : {len(sched_trips - et_trips):,}")

In [ ]:
cols_et = ["data_frame_ref", "journey_ref", "stop_ref", "occ", "visit_status", "visit_time", "aimed_time", "delay_s", "delay_outlier", "n_snapshots"]
cmp = exp_bus.merge(
    et_visits[cols_et],
    left_on=["service_date", "trip_id", "stop_id", "occ"],
    right_on=["data_frame_ref", "journey_ref", "stop_ref", "occ"],
    how="left",
)
trip_in_feed = cmp.set_index(["service_date", "trip_id"]).index.isin(list(et_trips))
cmp["final_status"] = np.where(
    ~trip_in_feed, "trip_not_in_feed",
    cmp["visit_status"].fillna("stop_not_in_feed"),
)

status_by_day = (cmp.groupby(["service_date", "final_status"]).size().unstack(fill_value=0))
status_by_day["scheduled_total"] = status_by_day.sum(axis=1)
status_by_day["observed_share"] = (status_by_day.get("observed", 0) / status_by_day["scheduled_total"]).round(3)
display(status_by_day)

**How to read this table**

- `observed` → scheduled and a bus really arrived/departed (cleaned "pure" visits)
- `cancelled` → the feed says it was cancelled
- `unconfirmed` → in the feed, but no actual time recorded
- `trip_not_in_feed` → timetable says it should run; real-time feed has nothing about that trip (coverage gap, not necessarily a missed bus)
- `stop_not_in_feed` → trip is in the feed but this stop is missing from it

In [ ]:
# Per stop: expected vs observed
per_stop = (cmp.groupby(["service_date", "stop_id"])
              .agg(expected_visits=("trip_id", "size"),
                   observed_visits=("final_status", lambda s: (s == "observed").sum()),
                   cancelled=("final_status", lambda s: (s == "cancelled").sum()),
                   median_delay_s=("delay_s", "median"))
              .reset_index()
              .merge(stops[["stop_id", "stop_name", "stop_lat", "stop_lon"]], on="stop_id", how="left"))
per_stop["observed_share"] = (per_stop["observed_visits"] / per_stop["expected_visits"]).round(3)
display(per_stop.sort_values("observed_visits", ascending=False).head(15))

In [ ]:
# Per route
per_route = (cmp.groupby(["service_date", "route_short_name"])
               .agg(expected_visits=("trip_id", "size"),
                    observed_visits=("final_status", lambda s: (s == "observed").sum()))
               .reset_index())
per_route["observed_share"] = (per_route["observed_visits"] / per_route["expected_visits"]).round(3)
display(per_route.sort_values("expected_visits", ascending=False).head(15))

# Quick charts
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
status_by_day.drop(columns=["scheduled_total", "observed_share"]).plot(kind="bar", stacked=True, ax=ax[0], title="Scheduled stop events by outcome")
obs = cmp[cmp["final_status"] == "observed"]["delay_s"].dropna() / 60
obs[(obs > -10) & (obs < 30)].plot(kind="hist", bins=40, ax=ax[1], title="Delay of observed visits (minutes, -10 to +30)")
plt.tight_layout(); plt.show()

## 8. Save the clean tables

- `clean_stop_visits.parquet` — **one row per confirmed bus visit** (stop, time, delay, route). This is the "pure" table to build on.
- `stop_event_status.parquet` — every scheduled bus stop event with its outcome (expected vs observed).
- `expected_per_stop.parquet` — expected visits per stop per day from the static timetable.

In [ ]:
clean_cols = ["data_frame_ref", "journey_ref", "line_ref", "route_short_name", "route_type_name", "direction_ref",
              "stop_order", "stop_ref", "stop_name", "stop_lat", "stop_lon",
              "aimed_arrival", "actual_arrival", "aimed_departure", "actual_departure",
              "visit_time", "aimed_time", "delay_s", "delay_outlier", "n_snapshots"]
clean_stop_visits = et_visits[et_visits["visit_status"] == "observed"][clean_cols].rename(columns={"data_frame_ref": "service_date"})

clean_stop_visits.to_parquet(OUT_DIR / "clean_stop_visits.parquet", index=False)
cmp.to_parquet(OUT_DIR / "stop_event_status.parquet", index=False)
expected_per_stop.to_parquet(OUT_DIR / "expected_per_stop.parquet", index=False)

print(f"clean_stop_visits: {len(clean_stop_visits):,} rows")
display(clean_stop_visits.head())

## Next steps

1. Run Part 4.3 on the real data and look at how full `actual_*` really is. If `observed` is far below what you expect, tell me the value counts of `arrival_status` / `departure_status` and we adjust the rule in 6.4.
2. Check `trip_not_in_feed`: if large, the feed doesn't cover everything (e.g. some operators/contracts), and "expected vs observed" needs to be restricted to covered trips.
3. Decide how to treat outliers (`delay_outlier`) and the first/last stop of each trip (only a departure or only an arrival exists).
4. Note the clocks change on **2026-10-25** (03:00 → 02:00 in Norway). All times here are tz-aware, so it works, but any hand-written hour bucketing should use the local tz-aware time.